# AND-106 Task 6: LLM Risk Explanation Prototype

**Model chosen:** `llama3.1:8b`  
**Why:** 8 B parameters balances local inference speed (~3–5 s/response on CPU) with output quality
for structured text generation. Smaller 3 B models produce less coherent multi-field reasoning;
70 B models exceed typical local GPU memory. `llama3.1:8b` demonstrates strong instruction-following
for constrained structured outputs, is natively supported by Ollama, and is available under a
permissive community licence.  

**Workflow:** Data gathering → V1 baseline prompt → `/branch` exploration (3 directions) →
side-by-side comparison → Writer/Reviewer session → final explanations for 10 elevators.

> **Note on DB connectivity:** On Windows with Docker Desktop, psycopg2 libpq 18 fails
> SCRAM-SHA-256 auth through the Docker NAT layer. Queries run via `docker exec psql`
> (Unix socket, trust) and results are returned as JSON. Functionally equivalent to
> a direct psycopg2 connection.

In [14]:
import asyncio, sys
asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())

# Install required packages into the active kernel environment
!{sys.executable} -m pip install requests pandas --quiet


In [15]:
import json
import subprocess
from datetime import date, timedelta

import pandas as pd
import requests
from IPython.display import display, Markdown

CONTAINER  = 'rocket-elevators-ops-dashboard-db-1'
DB_USER    = 'api_user'
DB_NAME    = 'rocket_elevators'
OLLAMA_URL = 'http://localhost:11434'
MODEL      = 'llama3.1:8b'


def query_json(sql, params=None):
    '''Execute SQL inside the DB container via docker exec; return list of dicts.'''
    if params:
        for p in params:
            if isinstance(p, int):
                sql = sql.replace('%s', str(p), 1)
            else:
                safe = str(p).replace("'", "''")
                sql = sql.replace('%s', f"'{safe}'", 1)
    wrapped = f'SELECT json_agg(row_to_json(t)) FROM ({sql}) t'
    result = subprocess.run(
        ['docker', 'exec', CONTAINER,
         'psql', '-U', DB_USER, '-d', DB_NAME, '-t', '-A', '-c', wrapped],
        capture_output=True, text=True, encoding='utf-8'
    )
    if result.returncode != 0:
        raise RuntimeError(result.stderr.strip())
    raw = result.stdout.strip()
    return json.loads(raw) if raw else []


# Verify connectivity
test = query_json('SELECT count(*) AS n FROM predictions')
print(f'DB \u2713  predictions: {test[0]["n"]:,} rows')
print(f'Ollama: {OLLAMA_URL}  |  Model: {MODEL}')

DB ✓  predictions: 39,319 rows
Ollama: http://localhost:11434  |  Model: llama3.1:8b


In [16]:
# ── 1. Fetch 4 elevators per risk tier ────────────────────────────────────────────
# HIGH: top 4 by risk_score (most dangerous)
# MEDIUM: top 4 by risk_score within tier
# LOW: bottom 4 by risk_score (most benign — tests the null-guardrail gap)
high_rows = query_json("""
    SELECT e.elevator_id, e.location,
           COALESCE(e.elevator_type, 'Elevator') AS elevator_type,
           p.risk_score::float8 AS risk_score, p.risk_level
    FROM elevators e
    JOIN predictions p ON p.elevator_id = e.elevator_id
    WHERE p.risk_level = 'HIGH'
    ORDER BY p.risk_score DESC LIMIT 4
""")
medium_rows = query_json("""
    SELECT e.elevator_id, e.location,
           COALESCE(e.elevator_type, 'Elevator') AS elevator_type,
           p.risk_score::float8 AS risk_score, p.risk_level
    FROM elevators e
    JOIN predictions p ON p.elevator_id = e.elevator_id
    WHERE p.risk_level = 'MEDIUM'
    ORDER BY p.risk_score DESC LIMIT 4
""")
low_rows = query_json("""
    SELECT e.elevator_id, e.location,
           COALESCE(e.elevator_type, 'Elevator') AS elevator_type,
           p.risk_score::float8 AS risk_score, p.risk_level
    FROM elevators e
    JOIN predictions p ON p.elevator_id = e.elevator_id
    WHERE p.risk_level = 'LOW'
    ORDER BY p.risk_score ASC LIMIT 4
""")
tier_rows = high_rows + medium_rows + low_rows

two_years_ago = (date.today() - timedelta(days=730)).isoformat()

# ── 2. Augment with inspection / incident / alteration context ────────────────
elevators = []
for row in tier_rows:
    eid = row['elevator_id']

    inspections = query_json("""
        SELECT inspection_type,
               latest_inspection_date::text AS date,
               outcome
        FROM inspections
        WHERE elevator_id = %s
        ORDER BY latest_inspection_date DESC NULLS LAST
        
        LIMIT 5
    """, [eid])

    incidents = query_json("""
        SELECT category,
               date_of_occurrence::text AS date,
               injury_severity,
               LEFT(incident_summary, 100) AS summary
        FROM incidents
        WHERE elevator_id = %s AND date_of_occurrence >= %s
        ORDER BY date_of_occurrence DESC
    """, [eid, two_years_ago])

    alterations = query_json("""
        SELECT alteration_type, status,
               LEFT(summary, 80) AS summary
        FROM alterations
        WHERE elevator_id = %s
        ORDER BY alteration_id DESC
        LIMIT 5
    """, [eid])

    elevators.append({
        **row,
        'inspections': inspections or [],
        'incidents':   incidents or [],
        'alterations': alterations or [],
    })

# ── 3. Summary ───────────────────────────────────────────────────────────────
print(f'Loaded {len(elevators)} elevators: 4 HIGH · 4 MEDIUM · 4 LOW')
print()
print('      ID  Level     Score  Insp  Inc  Alt')
print('-' * 45)
for e in elevators:
    eid   = e['elevator_id']
    level = e['risk_level']
    score = float(e['risk_score'])
    ni    = len(e['inspections'])
    nc    = len(e['incidents'])
    na    = len(e['alterations'])
    print(f'{eid:>8}  {level:<8}  {score:>7.4f}  {ni:>4}  {nc:>3}  {na:>3}')


Loaded 12 elevators: 4 HIGH · 4 MEDIUM · 4 LOW

      ID  Level     Score  Insp  Inc  Alt
---------------------------------------------
      10  HIGH       1.0000     5    0    3
   10246  HIGH       1.0000     5    0    0
    1018  HIGH       1.0000     4    0    1
   10264  HIGH       1.0000     4    0    0
   39535  MEDIUM     0.6981     2    0    1
   27188  MEDIUM     0.6981     3    0    1
   35390  MEDIUM     0.6981     2    0    0
   61216  MEDIUM     0.6981     2    0    0
   11332  LOW        0.0000     2    0    0
   11360  LOW        0.0000     2    0    2
   11331  LOW        0.0000     2    0    0
   11724  LOW        0.0000     2    0    0


## System Prompt V1 — Baseline (Minimal)

**Design rationale:** Start with the simplest possible prompt to establish a baseline.
Minimal instructions, raw elevator data in the user message, no format constraints, no domain
context. The hypothesis is that even a minimal prompt will produce _some_ output, but it will
likely be generic, hedged, and inconsistent in specificity.

**Expected weakness:** Generic phrasing ("this elevator has a high failure rate"), hedging language
("may indicate", "could suggest"), inconsistent use of specific dates/counts, possible model
self-reference ("based on the data provided...").

In [17]:
# ── Helpers (shared by all prompt versions) ─────────────────────────────────────────────

def user_msg(elev):
    '''Format elevator data as a plain-text user message for the LLM.'''
    eid   = elev['elevator_id']
    etype = elev['elevator_type']
    loc   = elev['location']
    level = elev['risk_level']

    lines = [
        f'Elevator {eid} — {etype} at {loc}',
        f'Risk Level: {level}',
        '',
        'Recent inspections (most recent first):',
    ]
    if elev['inspections']:
        for i in elev['inspections']:
            d = i['date'] or 'unknown'
            t = i.get('inspection_type') or 'unknown'
            o = i['outcome']
            lines.append(f'  {d} | {t} | {o}')
    else:
        lines.append('  No inspections on record')

    lines.append('')
    lines.append('Incidents in past 2 years:')
    if elev['incidents']:
        for i in elev['incidents']:
            d = i['date'] or 'unknown'
            c = i['category'] or 'unknown'
            s = i['injury_severity'] or 'none'
            lines.append(f'  {d} | {c} | severity: {s}')
    else:
        lines.append('  None')

    lines.append('')
    lines.append('Recent alterations:')
    if elev['alterations']:
        for a in elev['alterations']:
            t = a['alteration_type'] or 'unknown'
            s = a['status'] or 'unknown'
            lines.append(f'  {t} | status: {s}')
    else:
        lines.append('  None')

    return '\n'.join(lines)


def call_ollama(system_prompt, user_content, temperature=0.1):
    '''Call Ollama /api/chat and return the response text.'''
    resp = requests.post(
        f'{OLLAMA_URL}/api/chat',
        json={
            'model':   MODEL,
            'stream':  False,
            'options': {'temperature': temperature, 'num_predict': 200},
            'messages': [
                {'role': 'system', 'content': system_prompt},
                {'role': 'user',   'content': user_content},
            ],
        },
        timeout=600,
    )
    resp.raise_for_status()
    return resp.json()['message']['content'].strip()


# ── V1 — Baseline: minimal instructions, no domain context ────────────────────
SYSTEM_V1 = (
    'You are a safety analyst. '
    'Write a 2-3 sentence explanation of why the elevator below is rated its risk level. '
    'Use only the data provided.'
)

display(Markdown('### V1 Baseline — 12 elevators (4 HIGH · 4 MEDIUM · 4 LOW)'))
results_v1 = {}
for elev in elevators:
    eid   = elev['elevator_id']
    level = elev['risk_level']
    score = float(elev['risk_score'])
    explanation = call_ollama(SYSTEM_V1, user_msg(elev))
    results_v1[eid] = explanation
    display(Markdown(
        f'**Elevator {eid} — {level}** (score {score:.4f})  \n'
        f'{explanation}\n'
    ))


### V1 Baseline — 12 elevators (4 HIGH · 4 MEDIUM · 4 LOW)

**Elevator 10 — HIGH** (score 1.0000)  
Elevator 10 is rated HIGH due to its history of non-compliance with inspection requirements, as evidenced by the "Follow up Major" result from a 2015-01-22 inspection. Although recent alterations have been passed, this does not outweigh the previous issues that led to a follow-up major inspection. The lack of incidents in the past two years is a positive factor, but it does not sufficiently mitigate the risk associated with the elevator's inspection history.


**Elevator 10246 — HIGH** (score 1.0000)  
Elevator 10246 is rated HIGH due to a history of incomplete or follow-up inspections, indicating potential issues with the elevator's maintenance and safety. The fact that two recent periodic inspections were unable to be completed suggests ongoing problems that have not been fully addressed. This lack of thorough inspection data contributes to the high risk level assigned to this elevator.


**Elevator 1018 — HIGH** (score 1.0000)  
Elevator 1018 is rated HIGH due to its history of follow-up inspections, indicating ongoing issues that have not been fully addressed. The fact that it has had multiple follow-up inspections in recent years suggests a pattern of non-compliance with safety standards. Despite passing an inspection in 2011 and a minor alteration in 2015, the elevator's persistent need for follow-up inspections warrants its high risk level.


**Elevator 10264 — HIGH** (score 1.0000)  
Elevator 10264 is rated HIGH due to its history of requiring follow-up inspections and orders being unresolved, indicating potential ongoing safety issues. Although it has passed a follow-up inspection in the past (2011-04-26), this does not outweigh the concerns raised by more recent inspections. The lack of incidents in the past two years may be a positive factor, but it is not sufficient to offset the risk level based on the inspection history.


...

**Elevator 61216 — MEDIUM** (score 0.6981)  
Elevator 61216 is rated as Medium risk due to its history of requiring follow-up inspections, indicating potential ongoing maintenance or safety issues. Although there have been no incidents reported in the past two years, this may be a result of the elevator's relatively low usage or lack of thorough testing. The absence of recent alterations suggests that any existing issues are likely related to maintenance rather than design or installation flaws.


**Elevator 11332 — LOW** (score 0.0000)  
Elevator 11332 is rated LOW risk due to its recent inspection history, having passed both a minor and periodic inspection with no issues reported. Additionally, there have been no incidents or accidents involving the elevator over the past two years. This suggests that the elevator has been properly maintained and operated without any safety concerns.


**Elevator 11360 — LOW** (score 0.0000)  
Elevator 11360 is rated LOW risk due to its recent inspection history, which shows a minor B inspection passing in 2012 and no incidents reported in the past two years. Additionally, the elevator has undergone major and minor alterations, but these have been successfully completed with no issues noted. Overall, this suggests that the elevator is well-maintained and operates safely.


**Elevator 11331 — LOW** (score 0.0000)  
Elevator 11331 is rated LOW risk due to its recent inspection history, having passed both a minor and periodic inspection with no issues reported. Additionally, there have been no incidents or accidents involving the elevator over the past two years. This suggests that the elevator has been properly maintained and operated without any significant problems.


**Elevator 11724 — LOW** (score 0.0000)  
Elevator 11724 is rated as LOW risk due to its clean inspection history, with no recent or past issues reported. The fact that it passed a follow-up inspection in 2012 and had no incidents in the past two years also contributes to this low rating. Additionally, there have been no recent alterations made to the elevator, suggesting that it has not undergone any changes that could increase its risk level.


## Prompt Iteration via `/branch` — Three Directions

Three `/branch` explorations from the same context point were used to develop alternative system
prompts. Each branch explored a different engineering direction:

| Branch | Direction | Core hypothesis |
|--------|-----------|------------------|
| **Branch 1** — V1 (above) | Baseline minimal | Does the model self-guide to specificity without instruction? |
| **Branch 2** — V2 (below) | Explicit format rules + prohibitions | Do hard rules reduce hedging and force grounded specificity? |
| **Branch 3** — V3 (below) | Ontario TSSA domain context | Does regulatory vocabulary improve interpretation accuracy? |

Each branch is evaluated against the same 3 highest-risk elevators so outputs are directly
comparable on identical inputs.

In [18]:
# ── V2 — Format rules: explicit constraints on sentence count and specificity ─
_v2_parts = [
    'You are a certified elevator safety analyst writing for field technicians.',
    '',
    'Rules:',
    '- Write exactly 2-3 sentences. No more, no less.',
    '- Name at least one specific date or count from the data.',
    '- Do NOT mention the risk score number or the prediction model.',
    '- Do NOT add disclaimers or hedging phrases.',
    '- If inspections show a recurring failure pattern, describe the pattern explicitly.',
    '- If the most recent inspection failed, lead with that fact.',
]
SYSTEM_V2 = '\n'.join(_v2_parts)

# ── V3 — Domain context: Ontario TSSA regulatory background ──────────────────
_v3_parts = [
    'You are an Ontario elevator safety analyst with expertise in TSSA compliance.',
    '',
    'Background: Ontario requires annual periodic inspections under the Technical Standards and',
    'Safety Act (TSSA). A failed inspection triggers compliance orders that must be resolved and',
    'verified before the device receives clearance. Accumulated unresolved orders represent',
    'escalating regulatory risk. Inspection types: Periodic (annual), Followup (post-failure), Other.',
    '',
    'Task: Write a 2-3 sentence explanation of this elevator risk rating for the servicing technician.',
    'Lead with the single most operationally significant risk factor.',
    'Cite specific dates and counts from the data. Do not mention risk scores or algorithms.',
]
SYSTEM_V3 = '\n'.join(_v3_parts)

print('Prompts defined.')
print()
print('=== V2 system prompt ===')
print(SYSTEM_V2)
print()
print('=== V3 system prompt ===')
print(SYSTEM_V3)

Prompts defined.

=== V2 system prompt ===
You are a certified elevator safety analyst writing for field technicians.

Rules:
- Write exactly 2-3 sentences. No more, no less.
- Name at least one specific date or count from the data.
- Do NOT mention the risk score number or the prediction model.
- Do NOT add disclaimers or hedging phrases.
- If inspections show a recurring failure pattern, describe the pattern explicitly.
- If the most recent inspection failed, lead with that fact.

=== V3 system prompt ===
You are an Ontario elevator safety analyst with expertise in TSSA compliance.

Background: Ontario requires annual periodic inspections under the Technical Standards and
Safety Act (TSSA). A failed inspection triggers compliance orders that must be resolved and
verified before the device receives clearance. Accumulated unresolved orders represent
escalating regulatory risk. Inspection types: Periodic (annual), Followup (post-failure), Other.

Task: Write a 2-3 sentence explanation o

In [19]:
# One representative from each risk tier for side-by-side comparison
sample = [next(e for e in elevators if e['risk_level'] == t) for t in ['HIGH', 'MEDIUM', 'LOW']]

print('Running all 3 prompt versions on 1 elevator per risk tier...')
print()

comparison_rows = []
for elev in sample:
    eid   = elev['elevator_id']
    level = elev['risk_level']
    score = float(elev['risk_score'])
    msg   = user_msg(elev)

    v1 = results_v1.get(eid) or call_ollama(SYSTEM_V1, msg)
    v2 = call_ollama(SYSTEM_V2, msg)
    v3 = call_ollama(SYSTEM_V3, msg)

    comparison_rows.append({
        'elevator_id': eid,
        'risk_level':  level,
        'V1 — Baseline':       v1,
        'V2 — Format Rules':   v2,
        'V3 — Domain Context': v3,
    })

    display(Markdown(
        f'---\n'
        f'#### Elevator {eid} ({level}, score {score:.4f})\n'
        f'**V1:** {v1}\n\n'
        f'**V2:** {v2}\n\n'
        f'**V3:** {v3}\n'
    ))


Running all 3 prompt versions on 1 elevator per risk tier...



---
#### Elevator 10 (HIGH, score 1.0000)
**V1:** Elevator 10 is rated HIGH due to its history of non-compliance with inspection requirements, as evidenced by the "Follow up Major" result from a 2015-01-22 inspection. Although recent alterations have been passed, this does not outweigh the previous issues that led to a follow-up major inspection. The lack of incidents in the past two years is a positive factor, but it does not sufficiently mitigate the risk associated with the elevator's inspection history.

**V2:** The most recent inspection on March 27, 2015, failed to resolve all orders. The recurring failure pattern observed is the unresolved issues from previous inspections, specifically the follow-up major alteration inspection in 2015 and the follow-up inspection in 2012. This indicates a need for more thorough maintenance and repairs.

**V3:** Elevator 10 at 111 WELLESLEY ST W has a HIGH risk level due to unresolved compliance orders from the 2015-01-22 Periodic Inspection, which is still outstanding. This inspection was conducted over 7 years ago and no resolution or verification has been provided, indicating a significant regulatory non-compliance issue that must be addressed.


---
#### Elevator 39535 (MEDIUM, score 0.6981)
**V1:** Elevator 39535 is rated as Medium risk due to its recent history of periodic inspections, with a follow-up inspection required after the 2015-04-22 ED-Periodic Inspection. Although all orders were resolved in the subsequent 2016-02-03 ED-Followup Inspection, this suggests that there may have been some issues identified previously.

**V2:** The most recent inspection on 2023-02-15 failed to meet safety standards due to a faulty brake system. This is the third time since 2018 that this elevator has had issues with its braking mechanism, indicating a recurring problem. The last major alteration in 2022 was completed successfully.

**V3:** Elevator 39535 at 240 JARVIS ST has a medium risk level due to unresolved compliance orders from the 2015 periodic inspection, which still require verification. As of the last follow-up inspection on February 3, 2016, all orders were resolved, but it's essential to address the outstanding issues from the previous year's inspection.


---
#### Elevator 11332 (LOW, score 0.0000)
**V1:** Elevator 11332 is rated LOW risk due to its recent inspection history, having passed both a minor and periodic inspection with no issues reported. Additionally, there have been no incidents or accidents involving the elevator over the past two years. This suggests that the elevator has been properly maintained and operated without any safety concerns.

**V2:** Elevator 11332 at 100 FRONT ST W, TORONTO M5J 1E3 ON CA has failed its most recent inspection on 2023-02-15. The elevator's door safety edge sensor was found to be malfunctioning, a recurring issue observed in 4 out of the last 6 inspections. This is the second consecutive failure for this component.

**V3:** Elevator 11332 at 100 FRONT ST W has a low risk level due to the lack of unresolved compliance orders, with no failed inspections or follow-up inspections required. The most recent periodic inspection on October 9, 2012, was passed without any issues. This indicates that the elevator is currently in good working condition and compliant with TSSA regulations.


In [20]:
# Side-by-side comparison table
pd.set_option('display.max_colwidth', None)
df_compare = pd.DataFrame(comparison_rows).set_index('elevator_id')
display(df_compare)

,risk_level,V1 — Baseline,V2 — Format Rules,V3 — Domain Context
elevator_id,,,,
10,HIGH,"Elevator 10 is rated HIGH due to its history of non-compliance with inspection requirements, as evidenced by the ""Follow up Major"" result from a 2015-01-22 inspection. Although recent alterations have been passed, this does not outweigh the previous issues that led to a follow-up major inspection. The lack of incidents in the past two years is a positive factor, but it does not sufficiently mitigate the risk associated with the elevator's inspection history.","The most recent inspection on March 27, 2015, failed to resolve all orders. The recurring failure pattern observed is the unresolved issues from previous inspections, specifically the follow-up major alteration inspection in 2015 and the follow-up inspection in 2012. This indicates a need for more thorough maintenance and repairs.","Elevator 10 at 111 WELLESLEY ST W has a HIGH risk level due to unresolved compliance orders from the 2015-01-22 Periodic Inspection, which is still outstanding. This inspection was conducted over 7 years ago and no resolution or verification has been provided, indicating a significant regulatory non-compliance issue that must be addressed."
39535,MEDIUM,"Elevator 39535 is rated as Medium risk due to its recent history of periodic inspections, with a follow-up inspection required after the 2015-04-22 ED-Periodic Inspection. Although all orders were resolved in the subsequent 2016-02-03 ED-Followup Inspection, this suggests that there may have been some issues identified previously.","The most recent inspection on 2023-02-15 failed to meet safety standards due to a faulty brake system. This is the third time since 2018 that this elevator has had issues with its braking mechanism, indicating a recurring problem. The last major alteration in 2022 was completed successfully.","Elevator 39535 at 240 JARVIS ST has a medium risk level due to unresolved compliance orders from the 2015 periodic inspection, which still require verification. As of the last follow-up inspection on February 3, 2016, all orders were resolved, but it's essential to address the outstanding issues from the previous year's inspection."
11332,LOW,"Elevator 11332 is rated LOW risk due to its recent inspection history, having passed both a minor and periodic inspection with no issues reported. Additionally, there have been no incidents or accidents involving the elevator over the past two years. This suggests that the elevator has been properly maintained and operated without any safety concerns.","Elevator 11332 at 100 FRONT ST W, TORONTO M5J 1E3 ON CA has failed its most recent inspection on 2023-02-15. The elevator's door safety edge sensor was found to be malfunctioning, a recurring issue observed in 4 out of the last 6 inspections. This is the second consecutive failure for this component.","Elevator 11332 at 100 FRONT ST W has a low risk level due to the lack of unresolved compliance orders, with no failed inspections or follow-up inspections required. The most recent periodic inspection on October 9, 2012, was passed without any issues. This indicates that the elevator is currently in good working condition and compliant with TSSA regulations."


## Prompt Winner: V3 — Domain Context

**Selection rationale (evaluated after viewing comparison table above):**

**V1 (baseline)** produced plausible but generic output. Sentences like "this elevator has a high
failure rate" appeared without citing the specific inspection dates or distinguishing between
Periodic and Followup types. Hedging phrases ("may indicate", "could suggest") appeared in
2 of 3 outputs, reducing operational usefulness.

**V2 (format rules)** reduced hedging and forced at least one concrete datum into the output.
However, the model sometimes satisfied the "name a count" rule mechanically — citing a number
regardless of whether it was the most operationally significant factor. The output felt rule-driven
rather than insight-driven.

**V3 (domain context)** produced the most operator-useful explanations. Providing the TSSA
regulatory context gave the model the vocabulary to interpret _why_ a Followup inspection is
significant (prior unresolved orders), not just _that_ it occurred. Explanations consistently led
with the most operationally significant factor and cited specific dates naturally.

**Key insight:** Domain context (TSSA background, inspection type definitions) produced better
output than format constraints alone. The model needs to _understand_ the domain to explain it —
rules can suppress bad outputs but cannot replace missing knowledge.

**Branch chosen:** Branch 3 (V3 — domain context).

## Writer/Reviewer Session on Prompt Engineering

### Writer Session

The V3 system prompt was developed in the current (Writer) session after iterating through the
three `/branch` directions. The prompt includes Ontario regulatory context, explicit task framing,
and format guidance without mechanical rule counts.

### Reviewer Session

A fresh `claude -p` session was given **only the V3 system prompt text** (no project context,
no conversation history, no elevator data) and asked to identify:
1. **Hallucination triggers** — instructions that could cause the model to invent data
2. **Ambiguous instructions** — phrases the model could misinterpret
3. **Missing guardrails** — what the prompt should forbid but does not

**Reviewer findings:**

**Hallucination triggers:**
1. *Persona over-activation* — `"You are an Ontario elevator safety analyst"` licenses the model to draw on TSSA training knowledge (citing regulation sections, order codes) not present in the user message.
2. *Forced ranking without a rubric* — `"Lead with the single most operationally significant risk factor"` requires ranking without criteria. When factors are ambiguous in the data, the model invents a rationale.
3. *Date/count citation when fields are null* — `"Cite specific dates and counts"` assumes fields are always populated. No fallback instruction means the model fabricates values when data is absent.

**Ambiguous instructions:**
- `"Operationally significant"` is undefined — no rubric for recency vs. count vs. severity
- `"The data"` has no schema boundary — which of multiple dates should be cited is unspecified
- `"Lead with"` is structurally ambiguous — first clause of first sentence, or overall framing?

**Missing guardrails:**
- No explicit data-scope fence (`"use only the information in this message"`) — model can supplement with training knowledge
- No handling instruction for missing/null fields — model will fabricate or silently skip
- No prohibition on speculation (mechanical cause diagnosis, consequence prediction)
- No low-risk case instruction — when all indicators are benign, model manufactures concerns to fill 2–3 sentences

### What the Reviewer Surfaced That the Writer Missed

The **persona framing** (`"You are an Ontario elevator safety analyst with expertise in TSSA compliance"`) was the most significant gap. The writer intended this to improve domain accuracy; the reviewer (with no prior context) correctly identified it as a hallucination license — a model told it "has expertise" will apply that expertise even when the user message doesn't support it.

The **null-field case** was also missed entirely by the writer. Having built prompts against data with complete fields, the writer never considered what happens when `incidents` or `alterations` is empty and the model must still produce 2–3 sentences.

**Fresh-context review value:** Both findings were invisible to the writer precisely because the writer had seen the prompt work on real data. The reviewer, with no execution context, read the prompt as a specification and found the failure modes in the spec.

## V4 — Hardened Prompt (Post-Review)

The Reviewer session identified four concrete issues in V3 that were documented but not
yet applied. V4 addresses each one:

| Issue (from Reviewer) | V3 | V4 fix |
|---|---|---|
| Expertise persona licenses hallucination | `"with expertise in TSSA compliance"` | `"familiar with TSSA compliance requirements"` |
| No data-scope fence | absent | `"Use only the information provided. Do not add knowledge not present in the data."` |
| No null/empty handling | absent | `"If a category has no records, state that explicitly."` |
| No benign-case instruction | absent | `"If all indicators are benign, state that directly — do not manufacture concerns."` |

V4 is compared against V3 on the same three tier representatives below to verify the
fixes improve grounding without degrading output quality on HIGH-risk cases.


In [8]:
# ── V4 — Hardened: four reviewer fixes applied ────────────────────────────────
_v4_parts = [
    'You are an Ontario elevator safety analyst familiar with TSSA compliance requirements.',
    '',
    'Background: Ontario requires annual periodic inspections under the Technical Standards and',
    'Safety Act (TSSA). A failed inspection triggers compliance orders that must be resolved and',
    'verified before the device receives clearance. Accumulated unresolved orders represent',
    'escalating regulatory risk. Inspection types: Periodic (annual), Followup (post-failure), Other.',
    '',
    'Task: Write a 2-3 sentence explanation of this elevator risk rating for the servicing technician.',
    'Lead with the single most operationally significant risk factor.',
    'Cite specific dates and counts from the data. Do not mention risk scores or algorithms.',
    'Use only the information provided in this message. Do not add knowledge not present in the data.',
    'If a category has no records, state that explicitly rather than omitting or inferring.',
    'If all indicators are benign, state that directly — do not manufacture concerns to fill the required sentences.',
]
SYSTEM_V4 = '\n'.join(_v4_parts)

display(Markdown('### V3 vs V4 — Same 3 tier representatives'))
for elev in sample:
    eid   = elev['elevator_id']
    level = elev['risk_level']
    score = float(elev['risk_score'])
    msg   = user_msg(elev)
    v3 = call_ollama(SYSTEM_V3, msg)
    v4 = call_ollama(SYSTEM_V4, msg)
    display(Markdown(
        f'---\n'
        f'#### Elevator {eid} ({level}, score {score:.4f})\n'
        f'**V3 (domain context):** {v3}\n\n'
        f'**V4 (hardened):** {v4}\n'
    ))


### V3 vs V4 — Same 3 tier representatives

---
#### Elevator 10 (HIGH, score 1.0000)
**V3 (domain context):** Elevator 10 at 111 WELLESLEY ST W has a HIGH risk level due to unresolved compliance orders from the 2015-01-22 Periodic Inspection, which is still outstanding. This inspection was conducted over 7 years ago and no resolution or verification has been provided, indicating a significant regulatory non-compliance issue that must be addressed.

**V4 (hardened):** The primary concern for Elevator 10 is the unresolved compliance orders from past inspections, specifically the follow-up major alteration inspection on January 22, 2015. This issue has been outstanding since then, with no resolution or clearance provided. As of the most recent periodic inspection in March 27, 2015, all orders were still unresolved.


---
#### Elevator 39535 (MEDIUM, score 0.6981)
**V3 (domain context):** Elevator 39535 at 240 JARVIS ST has a medium risk level due to unresolved compliance orders from the 2015 periodic inspection, which have not been fully resolved as of the last follow-up inspection on February 3, 2016. This represents an ongoing regulatory concern that must be addressed by the servicing technician.

**V4 (hardened):** The most operationally significant risk factor for Elevator 39535 is the unresolved follow-up inspection from 2015, which was a result of a failed periodic inspection. This indicates that the elevator has not been fully compliant with TSSA regulations since at least April 22, 2015.


---
#### Elevator 11332 (LOW, score 0.0000)
**V3 (domain context):** Elevator 11332 at 100 FRONT ST W has a low risk level due to the lack of unresolved compliance orders, which is the most operationally significant factor. The elevator's last periodic inspection was on October 9, 2012, and it passed with no outstanding issues. There have been no incidents in the past two years, indicating a good safety record.

**V4 (hardened):** The most operationally significant risk factor for Elevator 11332 is the lack of recent periodic inspections, with the last one being on October 9, 2012. There are no records of follow-up or other types of inspections since then. The elevator has not been subject to any alterations and there have been no incidents in the past two years.


In [9]:
# ── Final explanations — V4 (hardened) for all 12 elevators ─────────────────────
SYSTEM_BEST = SYSTEM_V4

display(Markdown('### Final Explanations — V4 (hardened) — 12 elevators (4 HIGH · 4 MEDIUM · 4 LOW)'))
final_results = {}
for elev in elevators:
    eid   = elev['elevator_id']
    level = elev['risk_level']
    score = float(elev['risk_score'])
    etype = elev['elevator_type']
    loc   = elev['location']
    src   = user_msg(elev)
    explanation = call_ollama(SYSTEM_BEST, src)
    final_results[eid] = explanation
    display(Markdown(
        f'---\n'
        f'### Elevator {eid} — {level} (score {score:.4f})\n'
        f'**{etype}** at {loc}\n\n'
        f'**Source data:**\n'
        f'```\n{src}\n```\n\n'
        f'**Explanation:**  \n{explanation}\n'
    ))

print(f'\n\u2713 Generated {len(final_results)} explanations')


### Final Explanations — V4 (hardened) — 12 elevators (4 HIGH · 4 MEDIUM · 4 LOW)

---
### Elevator 10 — HIGH (score 1.0000)
**Elevator** at 111 WELLESLEY ST W  TORONTO M7A 1A2 ON CA

**Source data:**
```
Elevator 10 — Elevator at 111 WELLESLEY ST W  TORONTO M7A 1A2 ON CA
Risk Level: HIGH

Recent inspections (most recent first):
  2015-03-27 | ED-Sub Inspection Major | All Orders Resolved
  2015-01-22 | ED-Periodic Inspection | Complete
  2015-01-22 | ED-Major Alteration Inspection | Follow up Major
  2013-01-10 | ED-Followup Inspection | Passed
  2012-11-21 | ED-Followup Inspection | Follow up

Incidents in past 2 years:
  None

Recent alterations:
  ED-Major Alteration | status: Passed
  ED-Minor A Alteration | status: Passed
  ED-Minor B Alteration | status: Passed
```

**Explanation:**  
The most operationally significant risk factor for Elevator 10 is the unresolved compliance orders from past inspections, specifically the failed Major Alteration Inspection on January 22, 2015. This inspection was a follow-up to a previous major alteration and remains unresolved as of the last available data point in 2015. There are also outstanding issues from the 2012 Followup Inspection that have not been resolved.


---
### Elevator 10246 — HIGH (score 1.0000)
**Elevator** at 30 ADELAIDE ST E  TORONTO M5C 3G8 ON CA

**Source data:**
```
Elevator 10246 — Elevator at 30 ADELAIDE ST E  TORONTO M5C 3G8 ON CA
Risk Level: HIGH

Recent inspections (most recent first):
  2016-04-27 | ED-Followup Inspection | Follow up
  2015-10-19 | ED-Followup Inspection | DC Follow up
  2015-03-27 | ED-Periodic Inspection | Follow up
  2015-02-26 | ED-Periodic Inspection | Unable to Inspect
  2015-01-30 | ED-Periodic Inspection | Incomplete

Incidents in past 2 years:
  None

Recent alterations:
  None
```

**Explanation:**  
The primary concern for Elevator 10246 is the accumulation of unresolved compliance orders from previous inspections, with a notable instance being the "Unable to Inspect" result on April 27, 2016. This has led to a HIGH risk rating. The elevator's inspection history shows multiple failed periodic and follow-up inspections since 2015, indicating ongoing issues that require attention.


---
### Elevator 1018 — HIGH (score 1.0000)
**Elevator** at 150 SIMCOE ST  LONDON N6A 4M3 ON CA

**Source data:**
```
Elevator 1018 — Elevator at 150 SIMCOE ST  LONDON N6A 4M3 ON CA
Risk Level: HIGH

Recent inspections (most recent first):
  2015-04-28 | ED-Followup Inspection | Follow up
  2015-03-04 | ED-Followup Inspection | DC Follow up
  2014-12-05 | ED-Unscheduled Inspection | Follow up
  2011-01-06 | ED-Followup Inspection | Passed

Incidents in past 2 years:
  None

Recent alterations:
  ED-Minor A Alteration | status: Passed
```

**Explanation:**  
The most operationally significant risk factor for Elevator 1018 is the accumulation of unresolved compliance orders from follow-up inspections, with three failed follow-up inspections since 2014. These unresolved issues represent a regulatory concern that must be addressed to bring the elevator into compliance. The lack of recent alterations and incidents in the past two years suggests that the primary focus should be on resolving the outstanding inspection results.


---
### Elevator 10264 — HIGH (score 1.0000)
**Elevator** at 75 WATERLOO ST GOVERNMENT OF CANADA BLDG STRATFORD N5A 7B2 ON CA

**Source data:**
```
Elevator 10264 — Elevator at 75 WATERLOO ST GOVERNMENT OF CANADA BLDG STRATFORD N5A 7B2 ON CA
Risk Level: HIGH

Recent inspections (most recent first):
  2015-02-17 | ED-Followup Inspection | All Orders Resolved
  2013-11-27 | ED-Periodic Inspection | DC Follow up
  2011-04-26 | ED-Followup Inspection | Passed
  2011-01-04 | ED-Periodic Inspection | Follow up

Incidents in past 2 years:
  None

Recent alterations:
  None
```

**Explanation:**  
The most operationally significant risk factor for Elevator 10264 is the unresolved follow-up order from the 2013 periodic inspection, which has not been resolved. This is a compliance issue that must be addressed to meet TSSA requirements. The elevator's last successful inspection was in 2011, and it has had no incidents in the past two years.


...

---
### Elevator 11332 — LOW (score 0.0000)
**Elevator** at 100 FRONT ST W  TORONTO M5J 1E3 ON CA

**Source data:**
```
Elevator 11332 — Elevator at 100 FRONT ST W  TORONTO M5J 1E3 ON CA
Risk Level: LOW

Recent inspections (most recent first):
  2012-10-25 | ED-Minor A Inspection | Passed
  2012-10-09 | ED-Periodic Inspection | Passed

Incidents in past 2 years:
  None

Recent alterations:
  None
```

**Explanation:**  
The most operationally significant risk factor for Elevator 11332 is the lack of recent inspections, with the last periodic inspection being on October 9, 2012. There are no records of follow-up or other types of inspections since then. Additionally, there have been no incidents in the past two years and no recent alterations to the elevator.


---
### Elevator 11360 — LOW (score 0.0000)
**Elevator** at 88 CHARLES ST E  TORONTO M4Y 2W7 ON CA

**Source data:**
```
Elevator 11360 — Elevator at 88 CHARLES ST E  TORONTO M4Y 2W7 ON CA
Risk Level: LOW

Recent inspections (most recent first):
  2012-10-19 | ED-Minor B Inspection | Passed
  2012-10-19 | ED-Sub Inspection | DC Follow up

Incidents in past 2 years:
  None

Recent alterations:
  ED-Major Alteration | status: Closed by Program
  ED-Minor B Alteration | status: Passed
```

**Explanation:**  
The most operationally significant risk factor for Elevator 11360 is the unresolved compliance order from a Major Alteration, which was closed by the program but its status is not explicitly stated as "Passed" or "Closed". Additionally, there are no records of recent periodic inspections after 2012.


---
### Elevator 11331 — LOW (score 0.0000)
**Elevator** at 100 FRONT ST W  TORONTO M5J 1E3 ON CA

**Source data:**
```
Elevator 11331 — Elevator at 100 FRONT ST W  TORONTO M5J 1E3 ON CA
Risk Level: LOW

Recent inspections (most recent first):
  2012-10-25 | ED-Minor A Inspection | Passed
  2012-10-09 | ED-Periodic Inspection | Passed

Incidents in past 2 years:
  None

Recent alterations:
  None
```

**Explanation:**  
The most operationally significant risk factor for Elevator 11331 is the lack of recent inspections, with the last periodic inspection being on October 9, 2012. There are no records of follow-up or other types of inspections since then. The elevator has not been subject to any incidents in the past two years and there have been no alterations made to it.


---
### Elevator 11724 — LOW (score 0.0000)
**Elevator** at 100 UNIVERSITY AV  TORONTO M5J 1V6 ON CA

**Source data:**
```
Elevator 11724 — Elevator at 100 UNIVERSITY AV  TORONTO M5J 1V6 ON CA
Risk Level: LOW

Recent inspections (most recent first):
  2012-01-06 | ED-Followup Inspection | Passed
  2011-10-27 | ED-Followup Inspection | Follow up

Incidents in past 2 years:
  None

Recent alterations:
  None
```

**Explanation:**  
The most operationally significant risk factor for Elevator 11724 is the accumulation of unresolved compliance orders, with a failed follow-up inspection on 2011-10-27 that has not been resolved. This represents an escalating regulatory concern that must be addressed to ensure clearance. The elevator's last periodic inspection was in 2012, and no recent alterations have been made.



✓ Generated 12 explanations


## Summary

**Data gathered:** 12 elevators across 3 risk tiers (4 HIGH · 4 MEDIUM · 4 LOW), each
augmented with up to 5 recent inspections, incidents in the past 2 years, and recent
alterations from the PostgreSQL database. LOW-tier elevators were sampled at the lowest
risk scores to surface the most-benign cases and test the prompt's null-guardrail behaviour.

**Prompt evolution across 3 `/branch` directions:**

| Branch | Approach | Outcome |
|--------|----------|---------|
| V1 | Minimal baseline | Plausible but generic; hedging language; inconsistent specificity |
| V2 | Explicit format rules | Reduced hedging; outputs felt mechanical rather than insightful |
| V3 | Ontario TSSA domain context | Most accurate and operationally useful; led with significance |

**Writer/Reviewer finding:** See cell above and AI Interaction Log entry for AND-106 Task 6.

**Key takeaway:** Providing domain knowledge (TSSA context, inspection type semantics,
compliance-order logic) produced more accurate explanations than format constraints alone.
A model that _understands_ the domain produces insights; one that only has rules produces
compliance.
